In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

In [2]:
# ----------------------------------------------------------------------
# 1. FLEET DEFINITION
# ----------------------------------------------------------------------
EQUIPMENT_TYPES = {
    "Centrifugal Pump": {
        "count": 5,
        "base_vibration": 2.0,      # mm/s RMS
        "base_temp": 45,             # °C
        "base_pressure": 6.0,        # bar
        "base_rpm": 1750,
        "base_current": 18,          # Amps
        "failure_modes": ["Cavitation", "Bearing Wear", "Seal Failure"],
    },
    "Induction Motor": {
        "count": 5,
        "base_vibration": 1.5,
        "base_temp": 55,
        "base_pressure": None,       # not applicable
        "base_rpm": 2900,
        "base_current": 32,
        "failure_modes": ["Bearing Wear", "Winding Overheat", "Misalignment"],
    },
    "Air Compressor": {
        "count": 5,
        "base_vibration": 2.8,
        "base_temp": 60,
        "base_pressure": 8.0,
        "base_rpm": 1450,
        "base_current": 40,
        "failure_modes": ["Valve Failure", "Bearing Wear", "Overheat"],
    },
    "Conveyor Motor": {
        "count": 4,
        "base_vibration": 1.8,
        "base_temp": 48,
        "base_pressure": None,
        "base_rpm": 1150,
        "base_current": 22,
        "failure_modes": ["Misalignment", "Bearing Wear", "Belt Slippage"],
    },
}

START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2026, 6, 30)
READING_FREQ_HOURS = 6   # sensor reading every 6 hours


In [3]:
# ----------------------------------------------------------------------
# 2. BUILD EQUIPMENT ROSTER
# ----------------------------------------------------------------------
def build_roster():
    roster = []
    unit_id = 1
    for eq_type, cfg in EQUIPMENT_TYPES.items():
        for i in range(cfg["count"]):
            roster.append({
                "unit_id": f"EQ-{unit_id:03d}",
                "equipment_type": eq_type,
                "install_date": START_DATE - timedelta(days=int(np.random.uniform(200, 1200))),
                "base_vibration": cfg["base_vibration"] * np.random.uniform(0.9, 1.1),
                "base_temp": cfg["base_temp"] * np.random.uniform(0.95, 1.05),
                "base_pressure": cfg["base_pressure"] * np.random.uniform(0.95, 1.05) if cfg["base_pressure"] else None,
                "base_rpm": cfg["base_rpm"] * np.random.uniform(0.98, 1.02),
                "base_current": cfg["base_current"] * np.random.uniform(0.95, 1.05),
                "failure_modes": cfg["failure_modes"],
            })
            unit_id += 1
    return roster

In [4]:
# ----------------------------------------------------------------------
# 3. DEGRADATION + FAILURE EVENT SCHEDULING PER UNIT
# ----------------------------------------------------------------------
def schedule_failure_events(unit, sim_days):
    """
    Randomly schedule 2-5 failure events across the simulation window
    for a unit, each preceded by a degradation ramp of 14-28 days.
    After a failure, a maintenance reset occurs.
    """
    events = []
    day_cursor = np.random.uniform(30, 90)  # first degradation starts early-ish
    while day_cursor < sim_days - 20:
        ramp_length = np.random.randint(14, 29)   # days of degradation before failure
        failure_day = day_cursor + ramp_length
        if failure_day >= sim_days:
            break
        failure_mode = np.random.choice(unit["failure_modes"])
        events.append({
            "ramp_start_day": day_cursor,
            "failure_day": failure_day,
            "failure_mode": failure_mode,
        })
        # next ramp starts after a maintenance + healthy operation gap
        day_cursor = failure_day + np.random.uniform(25, 70)
    return events



In [5]:
# ----------------------------------------------------------------------
# 4. GENERATE SENSOR READINGS
# ----------------------------------------------------------------------
def generate_readings(roster):
    all_rows = []
    sim_days = (END_DATE - START_DATE).days
    timestamps = pd.date_range(START_DATE, END_DATE, freq=f"{READING_FREQ_HOURS}h")

    for unit in roster:
        events = schedule_failure_events(unit, sim_days)

        for ts in timestamps:
            day_num = (ts - START_DATE).total_seconds() / 86400

            # find if this timestamp falls within an active degradation ramp
            degradation_factor = 0.0   # 0 = healthy, 1 = at point of failure
            active_failure_mode = None
            days_to_failure = None
            in_failure_event = False

            for ev in events:
                if ev["ramp_start_day"] <= day_num <= ev["failure_day"]:
                    span = ev["failure_day"] - ev["ramp_start_day"]
                    progressed = day_num - ev["ramp_start_day"]
                    degradation_factor = (progressed / span) ** 1.5  # accelerating degradation curve
                    active_failure_mode = ev["failure_mode"]
                    days_to_failure = round(ev["failure_day"] - day_num, 2)
                    in_failure_event = True
                    break

                # --- sensor value generation ---
            noise = lambda scale: np.random.normal(0, scale)
            # vibration rises sharply with degradation
            vibration = unit["base_vibration"] * (1 + 1.8 * degradation_factor) + noise(0.08)

            # temperature rises moderately with degradation
            temperature = unit["base_temp"] * (1 + 0.35 * degradation_factor) + noise(0.5)

            # current draw rises as mechanical resistance increases
            current = unit["base_current"] * (1 + 0.25 * degradation_factor) + noise(0.3)

            # rpm drops slightly under mechanical strain
            rpm = unit["base_rpm"] * (1 - 0.05 * degradation_factor) + noise(5)

            # pressure: drops for cavitation, otherwise stable/slight rise
            if unit["base_pressure"] is not None:
                if active_failure_mode == "Cavitation":
                    pressure = unit["base_pressure"] * (1 - 0.4 * degradation_factor) + noise(0.05)
                else:
                    pressure = unit["base_pressure"] * (1 + 0.05 * degradation_factor) + noise(0.05)
            else:
                pressure = None

            # ambient context
            ambient_temp = 28 + 6 * np.sin(2 * np.pi * (day_num % 365) / 365) + noise(1.5)
            ambient_humidity = 70 + 10 * np.sin(2 * np.pi * (day_num % 365) / 365 + 1) + noise(3)

            # operating hours since last maintenance (proxy via degradation ramp position)
            hours_since_maint = (day_num - (events[0]["ramp_start_day"] if events else 0)) * 24 if events else day_num * 24
            hours_since_maint = max(hours_since_maint % 2000, 0)  # bounded proxy

            row = {
                "unit_id": unit["unit_id"],
                "equipment_type": unit["equipment_type"],
                "timestamp": ts,
                "vibration_mm_s": round(max(vibration, 0), 3),
                "temperature_c": round(temperature, 2),
                "current_a": round(max(current, 0), 2),
                "rpm": round(max(rpm, 0), 1),
                "pressure_bar": round(pressure, 3) if pressure is not None else None,
                "ambient_temp_c": round(ambient_temp, 2),
                "ambient_humidity_pct": round(max(min(ambient_humidity, 100), 0), 1),
                "hours_since_maintenance": round(hours_since_maint, 1),
                "days_to_failure": days_to_failure,          # null if healthy
                "failure_mode": active_failure_mode,          # null if healthy
                "will_fail_7d": 1 if (days_to_failure is not None and days_to_failure <= 7) else 0,
                "will_fail_14d": 1 if (days_to_failure is not None and days_to_failure <= 14) else 0,
                "is_failure_event": 1 if (days_to_failure is not None and days_to_failure <= 0.26) else 0,  # ~1 reading window
            }
            all_rows.append(row)

    return pd.DataFrame(all_rows)

In [14]:
from pathlib import Path

# ----------------------------------------------------------------------
# 5. RUN
# ----------------------------------------------------------------------
if __name__ == "__main__":
    print("Building equipment roster...")
    roster = build_roster()
    print(f"  {len(roster)} units across {len(EQUIPMENT_TYPES)} equipment types")

    print("Generating sensor readings (this may take a moment)...")
    df = generate_readings(roster)
    print(f"  Generated {len(df):,} rows")

    # quick sanity stats
    print("\nFailure event rate check:")
    print(df.groupby("equipment_type")["will_fail_7d"].mean().round(4))

    out_path = "C:\\Users\\user\\New Project\\predictive_maintenance\\sensor_data.csv"
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f"\nSaved to {out_path}")
    print(f"Columns: {list(df.columns)}")


Building equipment roster...
  19 units across 4 equipment types
Generating sensor readings (this may take a moment)...
  Generated 41,439 rows

Failure event rate check:
equipment_type
Air Compressor      0.0953
Centrifugal Pump    0.0899
Conveyor Motor      0.0995
Induction Motor     0.0926
Name: will_fail_7d, dtype: float64

Saved to C:\Users\user\New Project\predictive_maintenance\sensor_data.csv
Columns: ['unit_id', 'equipment_type', 'timestamp', 'vibration_mm_s', 'temperature_c', 'current_a', 'rpm', 'pressure_bar', 'ambient_temp_c', 'ambient_humidity_pct', 'hours_since_maintenance', 'days_to_failure', 'failure_mode', 'will_fail_7d', 'will_fail_14d', 'is_failure_event']


In [15]:
roster = build_roster()
pd.DataFrame(roster).head()

,unit_id,equipment_type,install_date,base_vibration,base_temp,base_pressure,base_rpm,base_current,failure_modes
0,EQ-001,Centrifugal Pump,2022-02-02,2.008004,45.784840,5.998400,1767.359285,18.278467,"[Cavitation, Bearing Wear, Seal Failure]"
1,EQ-002,Centrifugal Pump,2024-04-23,2.170268,43.792965,5.974195,1756.485109,18.539820,"[Cavitation, Bearing Wear, Seal Failure]"
2,EQ-003,Centrifugal Pump,2022-12-04,2.129557,44.730108,6.278808,1717.809831,18.030308,"[Cavitation, Bearing Wear, Seal Failure]"
3,EQ-004,Centrifugal Pump,2023-07-27,2.020799,44.220319,6.004171,1744.041485,17.945425,"[Cavitation, Bearing Wear, Seal Failure]"
4,EQ-005,Centrifugal Pump,2023-04-20,1.919215,44.647774,6.008384,1741.334205,17.351883,"[Cavitation, Bearing Wear, Seal Failure]"


In [16]:
pd.DataFrame(roster).dtypes

unit_id                   object
equipment_type            object
install_date      datetime64[ns]
base_vibration           float64
base_temp                float64
base_pressure            float64
base_rpm                 float64
base_current             float64
failure_modes             object
dtype: object

In [17]:
pd.DataFrame(roster).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   unit_id         19 non-null     object        
 1   equipment_type  19 non-null     object        
 2   install_date    19 non-null     datetime64[ns]
 3   base_vibration  19 non-null     float64       
 4   base_temp       19 non-null     float64       
 5   base_pressure   10 non-null     float64       
 6   base_rpm        19 non-null     float64       
 7   base_current    19 non-null     float64       
 8   failure_modes   19 non-null     object        
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 1.5+ KB
